# Task Decomposition

Decompose how models divide complex tasks across components. Identify which heads/MLPs
specialize for which subtasks, measure cooperation between components, and analyze
difficulty-sensitive computation.

## Why This Matters

Composition task decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.task_decomposition import (
    subtask_identification,
    functional_specialization,
    task_component_alignment,
    component_cooperation_analysis,
    task_difficulty_decomposition,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
metric_fns = {
    "task_a": lambda logits: float(logits[-1, 0] - logits[-1, 1]),
    "task_b": lambda logits: float(logits[-1, 2] - logits[-1, 3]),
}
print("Model ready")

## Subtask Identification

Identify which components contribute to which subtasks by measuring effects
of ablating each head and MLP on multiple metric functions.

In [ ]:
result = subtask_identification(model, tokens, metric_fns)
print("Primary subtask per component:")
for comp, task in list(result["primary_subtask"].items())[:8]:
    print(f"  {comp}: {task}")
print(f"\nSubtask->components mapping:")
for task, comps in result["subtask_components"].items():
    print(f"  {task}: {len(comps)} components")

## Functional Specialization

Measure how specialized each attention head and MLP layer is for a given task.
Higher specialization scores indicate components that contribute disproportionately.

In [ ]:
result = functional_specialization(model, tokens, metric_fn)
print(f"Attention effects shape: {result['attn_effects'].shape}")
print(f"MLP effects shape: {result['mlp_effects'].shape}")
print(f"\nMost specialized component: {result['most_specialized']}")
print(f"\nSpecialization scores (top 5):")
sorted_spec = sorted(result["specialization_scores"].items(), key=lambda x: -x[1])
for comp, score in sorted_spec[:5]:
    print(f"  {comp}: {score:.4f}")

## Task-Component Alignment

Build an alignment matrix showing how each component maps to each task.
Also measures task overlap (do tasks share components?) and component selectivity.

In [ ]:
result = task_component_alignment(model, tokens, metric_fns)
print(f"Alignment matrix: {len(result['alignment_matrix'])} components")
print(f"Task overlap: {result['task_overlap']:.4f}")
print(f"\nComponent selectivity (top 5):")
sorted_sel = sorted(result["component_selectivity"].items(), key=lambda x: -x[1])
for comp, sel in sorted_sel[:5]:
    print(f"  {comp}: {sel:.4f}")

## Component Cooperation Analysis

Analyze whether components work together (cooperate) or independently by comparing
joint ablation effects to sum of individual effects.

In [ ]:
result = component_cooperation_analysis(model, tokens, metric_fn)
print(f"Mean cooperation: {result['mean_cooperation']:.4f}")
print(f"Number of individual effects: {len(result['individual_effects'])}")
print(f"Number of pair effects: {len(result['pair_effects'])}")
print(f"\nCooperation scores (top 5 pairs):")
sorted_coop = sorted(result["cooperation_scores"].items(), key=lambda x: -abs(x[1]))
for pair, score in sorted_coop[:5]:
    sign = "cooperative" if score > 0 else "redundant"
    print(f"  {pair}: {score:.4f} ({sign})")

## Task Difficulty Decomposition

Compare component contributions on easy vs hard inputs to identify
which components are difficulty-sensitive (contribute more on hard tasks).

In [ ]:
tokens_easy = jnp.array([0, 5, 10, 15, 20, 25, 30])
tokens_hard = jnp.array([1, 2, 3, 4, 5, 6, 7])
result = task_difficulty_decomposition(model, tokens_easy, tokens_hard, metric_fn)
print(f"Difficulty-sensitive components: {len(result['difficulty_sensitive'])}")
print(f"Difficulty-insensitive components: {len(result['difficulty_insensitive'])}")
print(f"\nDifficulty-sensitive:")
for comp in result["difficulty_sensitive"][:5]:
    easy = result['easy_effects'][comp]
    hard = result['hard_effects'][comp]
    print(f"  {comp}: easy={easy:.4f}, hard={hard:.4f}")